In [ ]:
from IPython.display import clear_output, Markdown, display

In [ ]:
# No need to run this on colab. These libraries come pre-installed on colab
# %pip install torch torchvision torchaudio

# Content:

In this demo, we will take a look at the llama-v2 language model.

To use llama-v2, we will use the llama-cpp-python library, which is a python converted version of the llama-cpp library

For this, we need to install the library and to download the model weights file. The file can be downloaded from huggingface [repo](https://huggingface.co/TheBloke/Llama-2-7B-Chat-GGUF) of [TheBloke](https://huggingface.co/TheBloke). Credits to him for quantizing the model, saving it in different formats like GGML and GGUF and sharing with the community. He has a lot of other models on his channel that you can check out, including different versions of llama (like the 70B param sized one and code llama etc)

## Downloading model file

In [ ]:
!wget https://huggingface.co/TheBloke/Llama-2-7B-Chat-GGUF/resolve/main/llama-2-7b-chat.Q5_K_M.gguf

clear_output()

## Installing llama-cpp-python

installing supports different versions of hardware acceleration.

We will go with Cuda. Checkout the [Github Repo](https://github.com/abetlen/llama-cpp-python) for more options

In [ ]:
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install llama-cpp-python==0.2.74  # This takes a few mins when building wheel. Be patient.

## Running Llama-v2

In [ ]:
import json

from llama_cpp import Llama

In [ ]:
model = Llama(
    "llama-2-7b-chat.Q5_K_M.gguf",
    n_gpu_layers=-1, # To use GPU
    n_ctx=2048, # default ctx window is 512

    # llama-cpp supports multiple variations of llama and they can have different prompt format.
    # specifying the chat format isn't necessary but will let us use the create_chat_completions function.
    # alternatively we can just restructure our prompt and use the simple __call__
    chat_format="llama-2",
)

# clear_output()

Let's try a code generation example.

Note that llama-v2 and llama-v2 chat isn't specifically trained for code generations but it has some capability for it

Models trained for code generation are what's used by applications like github copilot or codenium.

Llama-2, being a text generation model can be trained and used for that purpose as well.
There is even a version of llama-2 called codellama, build for code generation.

The model file for codellama can also be found at TheBloke's repo.

In [ ]:
sys_msg = "You are a Python coding instructor. Help your students with their code."
user_prompt = """
Write me a function to calculate the fibbonaci series uptil length N.
Do NOT use recursion. Instead, use dynamic programming."""

prompt = f"""
[INST] <<SYS>>
{sys_msg}
<</SYS>>
{user_prompt}[/INST]
"""

In [ ]:
output = model(
    prompt,
    max_tokens=None  # sets no length limit
)

In [ ]:
output

In [ ]:
output["choices"][0]

In [ ]:
# response
display(Markdown(output["choices"][0]["text"]))

Let's try a translation example now



Like before, llama-v2 has some capability of translating but is not SPECIFICALLY trained for it

The translations here will vary in quality depending on what languages are being used.

The more of that language llama-v2 has seen during training, the better the results

For example, the translation of english to japanese will be better than from english to arabic.

instead of using llama-2-chat, it'd be better to use a finetuned seq2seq model for translations

In [ ]:
# Let's use create_chat_completions for now.
# Note that we need to specify the chat format for this to work correctly


sys_msg = "You are state of the are languge translator who is assisting people with translations"
user_prompt = """
Translate this English to Japanese:
'I woke up really early in the morning and then went for a jog to freshen up my mind'
"""

In [ ]:
output = model.create_chat_completion(
    messages = [
          {"role": "system", "content": sys_msg},
          {
              "role": "user",
              "content": user_prompt
          }
      ]
)

In [ ]:
output

In [ ]:
print(output['choices'][0]['message']['content'])

For the next example, let's ask it to solve a simple mathematical equation

for this example, we will also restrict it to produce JSON output which is more practical when the output needs to be further processed.

In [ ]:
sys_msg = """
You will be given reviews and your job is to analyse them, find and extract the negative parts.

1. Only output negative parts. If a review has no negative experiences in it, just output [].
2. One review can have multiple negative parts. find them all.
3. Do not alter parts of reviews or add something that isn't in the review.
4. Use the same keyword for the same issue across reviews. Do not use multiple keywords for the same issue.
5. Make sure the output is a valid JSON

Use this JSON format and dont say ANYHING other than this JSON:

[
    {
        "keyword-for-the-problem": "Part of the review mentioning the problem"
    }
]

"""

user_prompt = """
Analyse this review: The food was decent, but nothing extraordinary. It felt a bit overpriced for what we got. Service was a bit slow, but the staff was polite.
"""

prompt = f"""
[INST] <<SYS>>
{sys_msg}
<</SYS>>
{user_prompt}[/INST]
"""

In [ ]:
output = model(
    prompt,
    max_tokens=None  # sets no length limit
)

In [ ]:
output = json.loads(output["choices"][0]["text"])

In [ ]:
output

As most likely will be the case when you re-run this, the output has some issues.

Let's try **one shotting** by adding an example in the prompt

In [ ]:
sys_msg = """
You will be given reviews and your job is to analyse them, find and extract the negative parts.

1. Only output negative parts. If a review has no negative experiences in it, just output [].
2. One review can have multiple negative parts. find them all.
3. Do not alter parts of reviews or add something that isn't in the review.
4. Use the same keyword for the same issue across reviews. Do not use multiple keywords for the same issue.
5. Make sure the output is a valid JSON

Use this JSON format and dont say ANYHING other than this JSON:

[
    {
        "keyword-for-the-problem": "Part of the review mentioning the problem"
    }
]

Here is an example:

Review: `Decent food, but nothing outstanding. The service was average, and the atmosphere was a bit lacking. It's an okay option if you're in the area.`

[
  {
    "boring-ambience": "atmosphere was a bit lacking"
  }
]

"""

user_prompt = """

Review: `The food was decent, but nothing extraordinary. It felt a bit overpriced for what we got. Service was a bit slow, but the staff was polite.`
"""

prompt = f"""
[INST]<<SYS>>
{sys_msg}
<</SYS>>
{user_prompt}[/INST]
"""

output = model(
    prompt,
    max_tokens=None,  # sets no length limit
    stop=['<END>']
)

output = json.loads(output["choices"][0]["text"])

In [ ]:
output